# 06_03 - Decision Engine: Policy Evaluation
## 1. Methodology Overview

This notebook evaluates policy outputs with the shared project utilities for action frequency, coverage, reasons, and risk-context alignment.
It compares heuristic and RL decisions on the same validation period to support transparent decision-engine diagnostics.

**Source data used in this notebook:**
- `data/processed/train_features.csv`
- `data/processed/validation_features.csv`

**Core modules used in this notebook:**
- `src/models/train_model.py`
- `src/decision/policy_inputs.py`
- `src/decision/heuristic_policy.py`
- `src/rl/train_rl_agent.py`
- `src/decision/rl_policy.py`
- `src/decision/policy_evaluation.py`

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
if not (project_root / 'src').exists():
    project_root = Path('../../').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

train_df = pd.read_csv(project_root / 'data/processed/train_features.csv')
validation_df = pd.read_csv(project_root / 'data/processed/validation_features.csv')

print(f'Train features: {train_df.shape[0]} rows x {train_df.shape[1]} columns')
print(f'Validation features: {validation_df.shape[0]} rows x {validation_df.shape[1]} columns')

Train features: 1461 rows x 155 columns
Validation features: 366 rows x 155 columns


## 2. Build Heuristic and RL Decisions on a Common Input Set

To evaluate policies fairly, both engines are applied to the same policy-input dataframe derived from quantile forecasts.

In [2]:
from src.models.train_model import train_quantile_suite
from src.decision.policy_inputs import prepare_policy_inputs
from src.decision.heuristic_policy import apply_heuristic_policy
from src.rl.train_rl_agent import train_q_learning_agent
from src.decision.rl_policy import apply_rl_policy

quantile_output = train_quantile_suite(train_df=train_df, eval_df=validation_df)
policy_inputs_df = prepare_policy_inputs(validation_df, quantile_output.results)

heuristic_policy_df = apply_heuristic_policy(policy_inputs_df)

rl_training_artifacts = train_q_learning_agent(policy_inputs_df)
rl_policy_artifacts = apply_rl_policy(
    agent=rl_training_artifacts.agent,
    policy_inputs_df=policy_inputs_df,
    include_q_values=False,
)

# Build rl_policy_df from heuristic_policy_df so it already has the risk-signal
# columns (tail_vs_future_abs, tail_vs_central_abs, forecast_central, etc.)
# computed by _build_policy_signals.  We keep all those columns and overwrite
# the action column with RL decisions.
rl_policy_df = heuristic_policy_df.copy().reset_index(drop=True)
rl_policy_df['recommended_action'] = rl_policy_artifacts.decisions_df['recommended_action'].values
rl_policy_df['action_taken'] = rl_policy_df['recommended_action']
rl_policy_df['decision_reason'] = 'rl_policy'

2026-04-26 23:20:24 | INFO | src.rl.train_rl_agent | Starting RL training...
2026-04-26 23:20:24 | INFO | src.rl.train_rl_agent | Training dataframe shape: (337, 13)
2026-04-26 23:20:24 | INFO | src.rl.train_rl_agent | Episodes: 1000
2026-04-26 23:20:50 | INFO | src.rl.train_rl_agent | RL training completed successfully.
2026-04-26 23:20:50 | INFO | src.rl.train_rl_agent | Q-table states learned: 268
2026-04-26 23:20:50 | INFO | src.rl.train_rl_agent | Last episode reward: -11309.2550
2026-04-26 23:20:50 | INFO | src.rl.train_rl_agent | Best episode reward: -10969.1050
2026-04-26 23:20:50 | INFO | src.rl.train_rl_agent | Episode steps summary | mean=337.00 | last=337.00
2026-04-26 23:20:50 | INFO | src.rl.train_rl_agent | Reward summary:
   n_episodes   reward_mean   reward_std  reward_min  reward_max  reward_last  \
0        1000 -12859.089095  1467.036814  -19528.695  -10969.105   -11309.255   

   reward_rolling_mean_last_10  steps_mean  steps_last  q_table_states  \
0              

## 3. Evaluate Actions, Coverage, and Risk-Signal Alignment

We run the same evaluation helpers on both policy outputs to make diagnostics directly comparable.

In [3]:
from src.decision.policy_evaluation import (
    summarize_actions_by_calendar_context,
    summarize_actions_vs_risk_signals,
    summarize_policy_actions,
    summarize_policy_reasons,
    summarize_policy_time_coverage,
)

print('Heuristic policy: action summary')
display(summarize_policy_actions(heuristic_policy_df))

print('RL policy: action summary')
display(summarize_policy_actions(rl_policy_df))

print('Heuristic policy: time coverage')
display(summarize_policy_time_coverage(heuristic_policy_df))

print('RL policy: time coverage')
display(summarize_policy_time_coverage(rl_policy_df))

print('Heuristic policy: calendar context')
display(summarize_actions_by_calendar_context(heuristic_policy_df))

print('RL policy: calendar context')
display(summarize_actions_by_calendar_context(rl_policy_df))

print('Heuristic policy: actions vs risk signals')
display(summarize_actions_vs_risk_signals(heuristic_policy_df))

print('RL policy: actions vs risk signals')
display(summarize_actions_vs_risk_signals(rl_policy_df))

print('Heuristic policy: decision reasons')
display(summarize_policy_reasons(heuristic_policy_df))

Heuristic policy: action summary


,recommended_action,n_days,share
0,buy_m1_future,240,0.712166
1,do_nothing,65,0.192878
2,shift_production,32,0.094955


RL policy: action summary


,recommended_action,n_days,share
0,shift_production,288,0.854599
1,do_nothing,40,0.118694
2,buy_m1_future,9,0.026706


Heuristic policy: time coverage


...

,recommended_action,tail_vs_future_abs,tail_vs_future_rel,tail_vs_central_abs,tail_vs_central_rel,forecast_central,forecast_tail,current_m1_future
0,buy_m1_future,16.669963,0.319910,17.020208,0.631956,63.559255,80.579463,63.909500
1,do_nothing,4.867943,0.082170,11.780690,0.059645,62.651715,74.432405,69.564462
2,shift_production,3.335167,0.078029,24.167956,10.426262,46.744086,70.912042,67.576875


RL policy: actions vs risk signals


,recommended_action,tail_vs_future_abs,tail_vs_future_rel,tail_vs_central_abs,tail_vs_central_rel,forecast_central,forecast_tail,current_m1_future
0,buy_m1_future,18.253769,0.476110,15.345053,1.251621,50.153160,65.498213,47.244444
1,do_nothing,14.087317,0.449978,29.687128,10.140838,24.017688,53.704817,39.617500
2,shift_production,12.833875,0.216432,14.924926,0.251002,67.396901,82.321827,69.487951


Heuristic policy: decision reasons


,recommended_action,decision_reason,n_days
0,buy_m1_future,Tail risk exceeds futures price threshold,240
1,do_nothing,No rule triggered,65
2,shift_production,Weekend + high tail risk vs central forecast,32
